# Earthquakes Bronze — Auto Loader Pipeline

Ingests raw USGS earthquake GeoJSON snapshots from `/Volumes/weather_api/raw/raw_files/earthquakes/` into `weather_api.bronze.earthquakes_raw` using Auto Loader.

- Each JSON file is a single pretty-printed record (`multiLine=True`)
- Preserves the **complete GeoJSON feed**: feed-level `metadata`, `features` array, and `bbox`
- Schema is inferred and evolves automatically with `rescue` mode
- `_rescued_data` column captures any fields outside the inferred schema
- `availableNow` trigger processes all pending files then stops — ideal for scheduled runs
- Ingestion metadata: `_ingest_timestamp`, `_source_file_path`

In [0]:
from pyspark.sql.functions import current_timestamp, col

# ── Configuration ──
SOURCE_PATH = "/Volumes/weather_api/raw/raw_files/earthquakes"
SCHEMA_LOCATION = "/Volumes/weather_api/pipeline_state/autoloader_state/_schemas/earthquakes"
CHECKPOINT_LOCATION = "/Volumes/weather_api/pipeline_state/autoloader_state/_checkpoints/earthquakes"
TARGET_TABLE = "weather_api.bronze.earthquakes_raw"

# ── Create schema if needed ──
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_api.bronze")

# ── Auto Loader: read raw GeoJSON snapshots ──
# multiLine=True because files are pretty-printed (one JSON object per file)
# The complete GeoJSON is preserved: metadata (feed-level), features array, bbox
# schemaEvolutionMode=rescue captures new/changed fields in _rescued_data
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(SOURCE_PATH)
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file_path", col("_metadata.file_path"))
)

# ── Write to Delta table (incremental append) ──
# Each snapshot (file) becomes one row containing the full GeoJSON feed
# availableNow=True processes all pending files then stops the stream
# checkpointLocation ensures files are not reprocessed on restart
(raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
    .awaitTermination()
)

print(f"✅ Earthquakes Bronze pipeline complete — data written to {TARGET_TABLE}")

In [0]:
# ── Validation ──
from pyspark.sql.functions import size, col

print(f"Row count: {spark.table(TARGET_TABLE).count()}")
print("\nSchema:")
spark.table(TARGET_TABLE).printSchema()

print("\nFeed metadata (preserved from GeoJSON):")
spark.table(TARGET_TABLE).select(
    "metadata.source",
    "metadata.retrieval_timestamp_utc",
    "raw_response.metadata.title",
    "raw_response.metadata.count",
    "raw_response.metadata.status",
    "_ingest_timestamp",
    "_source_file_path",
).show(5, truncate=80)

print("\nFeature count per snapshot:")
spark.table(TARGET_TABLE).select(
    "_ingest_timestamp",
    size("raw_response.features").alias("feature_count"),
).show()

print("\nSample earthquake feature (first feature in each snapshot):")
spark.table(TARGET_TABLE).select(
    col("raw_response.features").getItem(0).getField("properties").getField("mag").alias("mag"),
    col("raw_response.features").getItem(0).getField("properties").getField("place").alias("place"),
    col("raw_response.features").getItem(0).getField("properties").getField("time").alias("time"),
    col("raw_response.features").getItem(0).getField("geometry").getField("coordinates").alias("coords"),
).show(5, truncate=80)

print("\nRescued data (should be null if all fields matched schema):")
spark.table(TARGET_TABLE).select(
    "_ingest_timestamp",
    "_rescued_data",
).show(5, truncate=80)